Séance : 1

Nom : OBOSSOU Richard

Porteur : P2

Etape assignée :
1.2 :Charger et compter
1.3 :Typer
1.5 :Valeur impossible
1.10 :Le plancher

In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit/Carnet_de_demarrage.ipynb
/kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit/Enonce.md
/kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit/HomeCredit_columns_description.csv
/kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit/a_decider.csv
/kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit/train.csv


In [6]:
 print(f"train : {tr.shape} | a_decider : {ad.shape} | dico application : {len(dico)} lignes")
print("colonnes en plus dans train :", set(tr.columns) - set(ad.columns))
print("doublons SK_ID_CURR train/a_decider :", tr.SK_ID_CURR.duplicated().sum(), ad.SK_ID_CURR.duplicated().sum())
print("ID communs train/a_decider :", len(set(tr.SK_ID_CURR) & set(ad.SK_ID_CURR)))
print("lignes entièrement dupliquées (train) :", tr.duplicated().sum())
print("colonnes hors dictionnaire :", set(tr.columns) - set(dico.Row.str.strip()))
print("TARGET :", tr.TARGET.value_counts().to_dict(), f"-> {tr.TARGET.mean():.4%} de défauts")
m = tr.isna().mean()
print(f"colonnes avec manquants : {(m>0).sum()} | >30 % : {(m>.3).sum()} | >50 % : {(m>.5).sum()} | max : {m.max():.1%}")

train : (246008, 122) | a_decider : (61503, 121) | dico application : 122 lignes
colonnes en plus dans train : {'TARGET'}
doublons SK_ID_CURR train/a_decider : 0 0
ID communs train/a_decider : 0
lignes entièrement dupliquées (train) : 0
colonnes hors dictionnaire : set()
TARGET : {0: 226148, 1: 19860} -> 8.0729% de défauts
colonnes avec manquants : 66 | >30 % : 50 | >50 % : 41 | max : 69.9%


In [7]:


from pathlib import Path
import numpy as np
import pandas as pd

FICHIERS = ["train.csv", "a_decider.csv", "HomeCredit_columns_description.csv"]

def trouver_dossier():
    for racine in [Path("/kaggle/input"), Path(".")]:
        if not racine.exists():
            continue
        for d in [racine, *racine.rglob("*")]:
            if d.is_dir() and all((d / f).is_file() for f in FICHIERS):
                return d
    raise FileNotFoundError(
        "Fichiers introuvables. Dans Kaggle : Add Input > Datasets > Your Datasets, puis relance. "
        "Fichiers vus dans /kaggle/input : "
        + str([str(p) for p in Path("/kaggle/input").rglob("*") if p.is_file()][:10])
    )

D = trouver_dossier()
print("dossier de données :", D)

tr = pd.read_csv(D / "train.csv")
ad = pd.read_csv(D / "a_decider.csv")
dico = pd.read_csv(D / "HomeCredit_columns_description.csv", encoding="latin-1")
dico = dico[dico.Table.str.startswith("application")]
print(tr.shape, ad.shape, len(dico))  # attendu : (246008, 122) (61503, 121) 122

dossier de données : /kaggle/input/datasets/richardobossou/home-credit1/Sujet_03_Home_Credit
(246008, 122) (61503, 121) 122


In [9]:
print(tr.dtypes.astype(str).value_counts().to_dict())
cat = tr.select_dtypes(exclude="number").columns.tolist()
print("catégorielles (", len(cat), ") :", cat)
flags = [c for c in tr.columns if c.startswith("FLAG_")]
bin_num = [c for c in flags if set(tr[c].dropna().unique()) <= {0, 1}]
print("FLAG_ numériques 0/1 :", len(bin_num), "| FLAG_ en texte Y/N :", [c for c in flags if c not in bin_num])
print("catégorielles à cardinalité > 10 :", {c: tr[c].nunique() for c in cat if tr[c].nunique() > 10})

{'float64': 65, 'int64': 41, 'object': 16}
catégorielles ( 16 ) : ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
FLAG_ numériques 0/1 : 26 | FLAG_ en texte Y/N : ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY']
catégorielles à cardinalité > 10 : {'OCCUPATION_TYPE': 18, 'ORGANIZATION_TYPE': 58}


In [10]:
for nom, df in (("train", tr), ("a_decider", ad)):
    emp = df.DAYS_EMPLOYED == 365243
    print(f"[{nom}] DAYS_EMPLOYED = 365243 (sentinelle) : {emp.sum()} ({emp.mean():.1%}) "
          f"| ORGANIZATION_TYPE='XNA' : {(df.ORGANIZATION_TYPE=='XNA').sum()} "
          f"| CODE_GENDER='XNA' : {(df.CODE_GENDER=='XNA').sum()}")
    print(f"[{nom}] DAYS_* > 0 : {sum((df[c]>0).sum() for c in df.filter(like='DAYS_') if c!='DAYS_EMPLOYED')} "
          f"| âge min/max : {-df.DAYS_BIRTH.max()/365.25:.1f}/{-df.DAYS_BIRTH.min()/365.25:.1f} "
          f"| montants <= 0 : {sum((df[c]<=0).sum() for c in ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE'])} "
          f"| EXT_SOURCE hors [0,1] : {sum(((df[c]<0)|(df[c]>1)).sum() for c in df.filter(like='EXT_SOURCE'))}")
    print(f"[{nom}] FAM_MEMBERS < enfants+1 : {(df.CNT_FAM_MEMBERS<df.CNT_CHILDREN+1).sum()} "
          f"| DEF_30 > OBS_30 : {(df.DEF_30_CNT_SOCIAL_CIRCLE>df.OBS_30_CNT_SOCIAL_CIRCLE).sum()}")

print("Valeurs extrêmes : revenu max", tr.AMT_INCOME_TOTAL.max(),
      "| enfants max", tr.CNT_CHILDREN.max(), "| OBS_30 max", tr.OBS_30_CNT_SOCIAL_CIRCLE.max(),
      "| AMT_REQ_BUREAU_QRT max", tr.AMT_REQ_CREDIT_BUREAU_QRT.max())
print("Taux de défaut si sentinelle DAYS_EMPLOYED :",
      tr.groupby(tr.DAYS_EMPLOYED == 365243).TARGET.mean().round(4).to_dict())

[train] DAYS_EMPLOYED = 365243 (sentinelle) : 44224 (18.0%) | ORGANIZATION_TYPE='XNA' : 44224 | CODE_GENDER='XNA' : 3
[train] DAYS_* > 0 : 0 | âge min/max : 21.0/69.0 | montants <= 0 : 0 | EXT_SOURCE hors [0,1] : 0
[train] FAM_MEMBERS < enfants+1 : 0 | DEF_30 > OBS_30 : 0
[a_decider] DAYS_EMPLOYED = 365243 (sentinelle) : 11150 (18.1%) | ORGANIZATION_TYPE='XNA' : 11150 | CODE_GENDER='XNA' : 1
[a_decider] DAYS_* > 0 : 0 | âge min/max : 20.5/69.1 | montants <= 0 : 0 | EXT_SOURCE hors [0,1] : 0
[a_decider] FAM_MEMBERS < enfants+1 : 0 | DEF_30 > OBS_30 : 0
Valeurs extrêmes : revenu max 117000000.0 | enfants max 19 | OBS_30 max 348.0 | AMT_REQ_BUREAU_QRT max 261.0
Taux de défaut si sentinelle DAYS_EMPLOYED : {False: 0.0866, True: 0.0542}


RATIO = 5  # HYPOTHÈSE à valider en équipe : un défaut accordé coûte 5 fois un bon client refusé

def cout(refus, df):
    fn = ((~refus) & (df.TARGET == 1)).sum()
    fp = (refus & (df.TARGET == 0)).sum()
    return (RATIO * fn + fp) / len(df), 1 - fn / df.TARGET.sum()

ext_moy = tr[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
idx = np.random.RandomState(0).permutation(len(tr))
A, B = idx[: len(tr) // 2], idx[len(tr) // 2:]
trA, trB = tr.iloc[A], tr.iloc[B]
extA, extB = ext_moy.iloc[A], ext_moy.iloc[B]

print(f"ratio FN/FP = {RATIO} | coût par demande, évalué sur la moitié B")
print("  tout accorder :", round(cout(pd.Series(False, index=trB.index), trB)[0], 3))
print("  tout refuser  :", round(cout(pd.Series(True, index=trB.index), trB)[0], 3))

seuil = min((cout(extA < t, trA)[0], t) for t in np.arange(.2, .6, .01))[1]
c, rappel = cout(extB < seuil, trB)
print(f"  refuser si moyenne EXT_SOURCE < {seuil:.2f} : coût {c:.3f}, "
      f"rappel des défauts {rappel:.1%}, refus {(extB < seuil).mean():.1%}")